# Catch the Eavesdropper (BB84)

*Secure the Grid, step 1 · UTC Problem 3*

Share a secret key over a simulated fiber link, then add an eavesdropper and catch them by the errors they cause.

**Your job:** fill in the encoding step and the eavesdropper, then report the error rate (QBER) with and without them.


In [ ]:
import numpy as np
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
sim = AerSimulator()
rng = np.random.default_rng(42)

n_bits = 100          # number of qubits Alice sends


In [ ]:
# Alice's random bits and random bases (0 = Z/standard, 1 = X/Hadamard)
alice_bits  = rng.integers(0, 2, n_bits)
alice_bases = rng.integers(0, 2, n_bits)
bob_bases   = rng.integers(0, 2, n_bits)


In [ ]:
def encode(bit, basis):
    """Return a 1-qubit circuit preparing Alice's bit in the chosen basis."""
    qc = QuantumCircuit(1, 1)
    # TODO: if bit == 1, apply X. If basis == 1 (X basis), apply H.
    raise NotImplementedError("Prepare the qubit for this bit and basis")
    return qc


In [ ]:
def measure(qc, basis):
    """Measure a 1-qubit circuit in the given basis; return the classical bit."""
    if basis == 1:
        qc.h(0)                     # rotate X basis back to Z before measuring
    qc.measure(0, 0)
    counts = sim.run(transpile(qc, sim), shots=1).result().get_counts()
    return int(max(counts, key=counts.get))


In [ ]:
def run_bb84(eavesdrop=False):
    bob_results = []
    for i in range(n_bits):
        qc = encode(alice_bits[i], alice_bases[i])
        if eavesdrop:
            # TODO: Eve measures in a RANDOM basis, then re-prepares and resends.
            # Hint: pick eve_basis = rng.integers(0,2); eve_bit = measure(qc, eve_basis);
            #       then qc = encode(eve_bit, eve_basis) before Bob measures.
            raise NotImplementedError("Implement the intercept-resend eavesdropper")
        bob_results.append(measure(qc, bob_bases[i]))
    return np.array(bob_results)


In [ ]:
def qber(bob_results):
    """Sift (keep matching bases) and return the error rate on the shared key."""
    keep = alice_bases == bob_bases
    a = alice_bits[keep]
    b = bob_results[keep]
    if len(a) == 0:
        return 0.0
    return np.mean(a != b)

clean = run_bb84(eavesdrop=False)
print("QBER, no eavesdropper:", round(qber(clean), 3))

eve = run_bb84(eavesdrop=True)
print("QBER, with eavesdropper:", round(qber(eve), 3), "  (expect ~0.25)")


### Hardware run: one job, not 100

The simulator cells above send one qubit per run, which is fine on a free local simulator. On real hardware that would be 100+ queued, billed jobs. Instead, put 16 BB84 rounds side by side in one circuit (qubit *k* carries round *k*) and run it once with 200 shots.

Every shot repeats the same 16 rounds. That's fine here: Part 3 asks for the error rate caused by hardware noise, and 200 shots × the sifted rounds gives you well over a thousand measured bits to estimate it.

In [ ]:
def bb84_batch_circuit(bits, bases, bob_bases):
    """One circuit with one BB84 round per qubit. Reuses your encode() from above."""
    k = len(bits)
    qc = QuantumCircuit(k, k)
    for q in range(k):
        qc.compose(encode(bits[q], bases[q]), qubits=[q], clbits=[q], inplace=True)
        if bob_bases[q] == 1:
            qc.h(q)                 # Bob measures in the X basis
    qc.measure(range(k), range(k))
    return qc


def qber_from_counts(counts, bits, bases, bob_bases):
    """Error rate on the sifted rounds, over every shot."""
    keep = [q for q in range(len(bits)) if bases[q] == bob_bases[q]]
    errors = total = 0
    for bitstring, c in counts.items():
        bob = bitstring[::-1]       # Qiskit order: qubit 0 is the LAST character
        for q in keep:
            errors += c * (int(bob[q]) != bits[q])
            total += c
    return errors / total


k = 16                              # rounds per circuit = qubits used
hw_args = (alice_bits[:k], alice_bases[:k], bob_bases[:k])
hw_qc = bb84_batch_circuit(*hw_args)

# Check it on the local simulator first (free)
counts = sim.run(transpile(hw_qc, sim), shots=200).result().get_counts()
print("Batched QBER on the local simulator:", round(qber_from_counts(counts, *hw_args), 3))

### Run it on a quantum computer through qBraid

Practice first on IonQ's simulator. It's free and can add realistic hardware noise. When your circuit works, switch the two lines to the real IonQ QPU for your hardware run. That run uses credits, so cap it at 200 shots and copy the job ID into your write-up.

In [ ]:
# --- Run on a quantum computer through qBraid ---
from qbraid.runtime import QbraidProvider

provider = QbraidProvider()   # in qBraid Lab this uses your account automatically

# 1) Practice for free: IonQ's simulator with a realistic noise model
device = provider.get_device("ionq:ionq:sim:simulator")
job = device.run(hw_qc, shots=200, runtime_options={"noise_model": "aria-1"})

# 2) Your real hardware run (uses credits). Swap these in when you're ready:
# device = provider.get_device("ionq:ionq:qpu:forte-1")
# job = device.run(hw_qc, shots=200)

print("Job ID:", job.id)          # copy this into your write-up
hw_counts = job.result().data.get_counts()
print(hw_counts)
print("QBER from this run:", round(qber_from_counts(hw_counts, *hw_args), 3))

## What to submit
- **Part 1:** a matching key with near-zero QBER (no eavesdropper).
- **Part 2:** the QBER with and without the eavesdropper (expect ~25%).
- **Part 3:** a hardware QBER, plus your threshold reasoning (in your write-up).
- **Part 4:** the written physics explanation.
